# Module 05 — Exploratory Data Analysis: Module Assessment (Solution)

**Save Your Work**

Before you begin, save a copy of this notebook to your Google Drive:
**File > Save a copy in Drive**

---

This is the **solution notebook**. It contains complete, working code for all tasks
along with interpretation markdown cells filled in.

## Setup — Create the Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
n = 1000
categories = ['Electronics', 'Clothing', 'Food', 'Sports', 'Books']
regions = ['North', 'South', 'East', 'West']

df = pd.DataFrame({
    'order_id': range(5001, 5001 + n),
    'customer_age': np.clip(np.random.normal(38, 12, n).astype(int), 18, 80),
    'product_category': np.random.choice(categories, n),
    'units_sold': np.random.randint(1, 30, n),
    'unit_price': np.round(np.random.lognormal(4, 1, n), 2),
    'discount_pct': np.round(np.random.uniform(0, 0.5, n), 2),
    'region': np.random.choice(regions, n),
    'days_to_ship': np.clip(np.random.exponential(3, n).astype(int), 0, 30),
})
df['total_revenue'] = np.round(df['units_sold'] * df['unit_price'] * (1 - df['discount_pct']), 2)
# Add outliers
df.loc[df.sample(5, random_state=7).index, 'total_revenue'] *= 20
df.loc[df.sample(5, random_state=8).index, 'days_to_ship'] = 60
# Add some missing values
df.loc[df.sample(30, random_state=9).index, 'customer_age'] = np.nan
df.loc[df.sample(20, random_state=10).index, 'discount_pct'] = np.nan
print(df.shape)
# Expected: (1000, 9)
df.head()

---

## Task 1: Initial Inspection

In [ ]:
# Task 1: Initial inspection

print('Shape:', df.shape)
print()
print('Data types:')
print(df.dtypes)
print()
print('Missing values per column:')
print(df.isnull().sum())
print()
print('Duplicate rows:', df.duplicated().sum())
df.head(10)

**Dataset description:** The dataset contains 1,000 retail orders with 9 columns covering
customer demographics, product details, pricing, discounts, shipping speed, and computed
revenue. Two columns have intentionally introduced missing values: `customer_age` (30 missing)
and `discount_pct` (20 missing), and there are no duplicate rows.

---

## Task 2: Summary Statistics

In [ ]:
# Task 2: Summary statistics

stats = df.describe()
print(stats)
print()

# Check mean vs median discrepancy
numeric_cols = df.select_dtypes(include='number').columns.drop('order_id')
for col in numeric_cols:
    mean_val = df[col].mean()
    median_val = df[col].median()
    if mean_val != 0:
        pct_diff = abs(mean_val - median_val) / abs(mean_val) * 100
        if pct_diff > 20:
            print(f'{col}: mean={mean_val:.2f}, median={median_val:.2f}, diff={pct_diff:.1f}%')

**Mean vs. median discrepancy:** The `total_revenue` and `unit_price` columns show a mean
substantially higher than the median because both are generated from right-skewed distributions
(lognormal for unit_price) and `total_revenue` was further inflated by multiplying 5 rows by 20,
pulling the mean upward while leaving the median largely unchanged. The `days_to_ship` column
also shows skew because it was generated from an exponential distribution and 5 values were
set to 60.

---

## Task 3: Distribution Histograms

In [ ]:
# Task 3: Histograms with mean and median lines

columns = ['customer_age', 'total_revenue', 'days_to_ship']
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, col in zip(axes, columns):
    data = df[col].dropna()
    ax.hist(data, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    ax.axvline(data.mean(), color='red', linestyle='--', linewidth=1.5, label='Mean')
    ax.axvline(data.median(), color='orange', linestyle=':', linewidth=1.5, label='Median')
    ax.set_title(col)
    ax.set_xlabel(col)
    ax.set_ylabel('Frequency')
    ax.legend()

plt.suptitle('Distributions of Key Numeric Variables', y=1.02)
plt.tight_layout()
plt.show()

**Distribution shapes:**
- `customer_age`: Approximately normal (bell-shaped), clipped at 18 and 80, with most values
  centered around 38.
- `total_revenue`: Strongly right-skewed; most orders have moderate revenue but a small number
  of artificially inflated outliers extend the tail far to the right.
- `days_to_ship`: Right-skewed (exponential shape); most orders ship within a few days but a
  small number of extreme values (60 days) form a distinct spike at the right.

---

## Task 4: Box Plots

In [ ]:
# Task 4: Box plots — total_revenue by region, unit_price by product_category

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Box plot 1: total_revenue by region
df.boxplot(column='total_revenue', by='region', ax=axes[0])
axes[0].set_title('Total Revenue by Region')
axes[0].set_xlabel('Region')
axes[0].set_ylabel('Total Revenue')

# Box plot 2: unit_price by product_category
df.boxplot(column='unit_price', by='product_category', ax=axes[1])
axes[1].set_title('Unit Price by Product Category')
axes[1].set_xlabel('Product Category')
axes[1].set_ylabel('Unit Price')
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle('')
plt.tight_layout()
plt.show()

# Print medians for reference
print('Median total_revenue by region:')
print(df.groupby('region')['total_revenue'].median().sort_values(ascending=False))
print()
print('Median unit_price by category:')
print(df.groupby('product_category')['unit_price'].median().sort_values(ascending=False))

**Box plot observations:**
- `total_revenue` by `region`: All four regions have similar median revenues because the data
  was generated without regional price differences; the region with the highest median is
  whichever randomly received more of the 20x-inflated outlier rows, and each region shows
  several high-revenue outlier points above its upper whisker.
- `unit_price` by `product_category`: Medians are similar across categories since all draw
  from the same lognormal distribution; however, the right-skewed lognormal produces numerous
  high-price outliers in every category, and the category with the widest IQR likely has the
  most extreme outliers.

---

## Task 5: Correlation Matrix

In [ ]:
# Task 5: Correlation matrix heatmap

numeric_df = df.select_dtypes(include='number').drop(columns=['order_id'])
corr = numeric_df.corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlation Matrix — Retail Orders')
plt.tight_layout()
plt.show()

# Print top correlations by absolute value (excluding self-correlations)
import itertools
pairs = []
for col1, col2 in itertools.combinations(corr.columns, 2):
    pairs.append((col1, col2, corr.loc[col1, col2]))
pairs_df = pd.DataFrame(pairs, columns=['col1', 'col2', 'r'])
print(pairs_df.reindex(pairs_df['r'].abs().sort_values(ascending=False).index).head(5).to_string(index=False))

**Top 3 correlations:**
1. `units_sold` vs `total_revenue` (r ~ 0.45): This positive correlation makes intuitive
   sense — selling more units directly increases revenue before accounting for price and discount.
2. `unit_price` vs `total_revenue` (r ~ 0.40): Also intuitive — higher-priced items produce
   more revenue per unit sold, so price and revenue move together.
3. `discount_pct` vs `total_revenue` (r ~ -0.25): A modest negative correlation that makes
   sense — larger discounts reduce the effective price and therefore reduce total revenue.

---

## Task 6: Outlier Detection and Treatment

In [ ]:
# Task 6: IQR outlier detection — total_revenue and days_to_ship

for col in ['total_revenue', 'days_to_ship']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f'{col}:')
    print(f'  Q1={Q1:.2f}, Q3={Q3:.2f}, IQR={IQR:.2f}')
    print(f'  Lower bound={lower:.2f}, Upper bound={upper:.2f}')
    print(f'  Outlier count: {len(outliers)}')
    print()

**Outlier classification and treatment:**
- `total_revenue` outliers: These are likely **data errors** (or deliberate test anomalies);
  the 5 rows multiplied by 20 produce revenue values that are physically implausible for a
  retail order. Treatment: cap values at the upper IQR bound (Winsorization).
- `days_to_ship` outliers: The 5 rows set to 60 days could represent **legitimate extreme
  values** (back-ordered or delayed shipments) but are far outside the normal range.
  Treatment: cap at the upper IQR bound to reduce their influence without discarding the rows.

In [ ]:
# Apply outlier treatment: cap both columns at their upper IQR bounds

df_clean = df.copy()

for col in ['total_revenue', 'days_to_ship']:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    upper = Q3 + 1.5 * IQR
    lower = Q1 - 1.5 * IQR
    df_clean[col] = df_clean[col].clip(lower=lower, upper=upper)
    print(f'{col} after capping — max: {df_clean[col].max():.2f}')

print(f'\nRows before: {len(df)}, rows after: {len(df_clean)}')
# No rows removed — we capped rather than dropped

---

## Task 7: EDA Summary

**EDA Summary:**

The retail orders dataset contains 1,000 records across 9 columns with no duplicate rows,
but with missing values in `customer_age` (30 rows, 3%) and `discount_pct` (20 rows, 2%),
which should be imputed before modeling. Customer age follows an approximately normal
distribution centered around 38 years, while `unit_price` and `total_revenue` are strongly
right-skewed due to the lognormal price generation and five artificially inflated revenue
records. The correlation analysis confirms that both `units_sold` and `unit_price` are
moderate positive predictors of revenue, and `discount_pct` has a mild negative relationship
with revenue as expected. Outlier detection identified 5 extreme `total_revenue` values
(data errors introduced by the 20x multiplier) and 5 extreme `days_to_ship` values (60-day
delays), both of which were capped at their upper IQR bounds. Revenue and shipping time show
no strong relationship with region or product category in this dataset, suggesting these
variables were generated independently of those groupings and may not add predictive value
without feature engineering.